# Train the flood model ensemble (Weeks 2 to 3)

This notebook trains **five U-Net models** on Sen1Floods11, one per random seed, and then measures:

1. **Accuracy:** each model, the ensemble, and the threshold baseline, on validation, test and the held-out Bolivia set.
2. **Uncertainty:** whether the ensemble's probabilities are more honest (calibrated) than a single model's.
3. **Best decision threshold:** the water/no-water cut-off that gives the best validation IoU.

Improvements over the first run: 50 epochs, a learning rate that slowly decreases (cosine schedule), Dice loss added to BCE (to find more of the water), the best epoch kept for each model, mixed precision for speed, and all data held in memory.

## How to run it on Kaggle (and leave it)

1. On kaggle.com: **Create > New Notebook**, then **File > Import Notebook**, and paste the GitHub link to this file.
2. In the right panel, **Session options**: Accelerator **GPU T4 x2** (not P100, which recent PyTorch no longer supports), and **Internet on** (needs a phone-verified Kaggle account).
3. Click **Save Version**, choose **Save & Run All (Commit)**, then **Save**. Kaggle runs it in the background; you can close the laptop.
4. Expected time: roughly 2 to 4 hours. When it finishes, open the version and download `outputs/` from the **Output** tab.

To check everything works before a long run, set `QUICK_TEST = True` in the next cell and run it interactively (a few minutes).

Version: 2026-10-28

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

In [ ]:
import os, sys, json, time, random, glob, re, subprocess

QUICK_TEST = False      # True = tiny run (few chips, 2 epochs) to check the notebook works

EPOCHS = 2 if QUICK_TEST else 50
BATCH = 2 if QUICK_TEST else 8
NUM_WORKERS = 0 if QUICK_TEST else 2
LR = 3e-4
SEEDS = [0, 1] if QUICK_TEST else [0, 1, 2, 3, 4]
PRETRAINED = os.environ.get("CUTOFF_PRETRAINED", "imagenet")   # ImageNet weights for the encoder
PRETRAINED = None if PRETRAINED.lower() == "none" else PRETRAINED

ON_KAGGLE = os.path.exists("/kaggle/working")
OUT = os.environ.get("CUTOFF_OUT", "/kaggle/working/outputs" if ON_KAGGLE else "/content/outputs")
DATA = os.environ.get("CUTOFF_DATA", "/tmp/sen1floods11")
REPO = os.environ.get("CUTOFF_REPO", "/tmp/cutoff-flood")
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
for d in (OUT, f"{OUT}/models", f"{OUT}/figures", DATA):
    os.makedirs(d, exist_ok=True)

# Project code: shared settings, preprocessing and metrics
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from src import config
from src.preprocess import preprocess_sar, clean_label
from src.metrics import Counts, Reliability
print("Outputs go to:", OUT)

## 1. Download the hand-labelled data

Downloads directly from the public Sen1Floods11 bucket over HTTPS (no Google login needed), with Google's `gsutil` tool as a fallback. Skipped if the files are already there.

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor

BASE = "https://storage.googleapis.com/sen1floods11/v1.1"
SPLITS = ["flood_train_data.csv", "flood_valid_data.csv", "flood_test_data.csv", "flood_bolivia_data.csv"]

def fetch(job):
    url, dest = job
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return
    for attempt in range(4):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            with open(dest, "wb") as f:
                f.write(r.content)
            return
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Could not download {url}: {last}")

def download_http():
    # 1. The four split files (known names), 2. every chip they list
    os.makedirs(f"{DATA}/flood_handlabeled", exist_ok=True)
    for name in SPLITS:
        fetch((f"{BASE}/splits/flood_handlabeled/{name}", f"{DATA}/flood_handlabeled/{name}"))
    chips = set()
    for name in SPLITS:
        for n in pd.read_csv(f"{DATA}/flood_handlabeled/{name}", header=None).iloc[:, 0].astype(str):
            chips.add(re.sub(r"_(S1Hand|LabelHand|S2Hand|S1Weak).*$", "", os.path.basename(n)))
    jobs = []
    for sub, suffix in [("S1Hand", "S1Hand"), ("LabelHand", "LabelHand")]:
        os.makedirs(f"{DATA}/{sub}", exist_ok=True)
        jobs += [(f"{BASE}/data/flood_events/HandLabeled/{sub}/{c}_{suffix}.tif", f"{DATA}/{sub}/{c}_{suffix}.tif")
                 for c in sorted(chips)]
    print(f"Downloading {len(jobs)} chip files ...")
    with ThreadPoolExecutor(16) as ex:
        list(ex.map(fetch, jobs))

def download_gsutil():
    # Fallback: Google's own tool, which can read public buckets without login
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gsutil"], check=True)
    for src in ["data/flood_events/HandLabeled/S1Hand", "data/flood_events/HandLabeled/LabelHand",
                "splits/flood_handlabeled"]:
        subprocess.run(["gsutil", "-m", "-q", "cp", "-r", f"gs://sen1floods11/v1.1/{src}", DATA], check=True)

import pandas as pd
if len(glob.glob(f"{DATA}/S1Hand/*.tif")) < 446:
    try:
        download_http()
    except Exception as e:
        print("Direct download failed, trying gsutil instead:", str(e)[:150])
        download_gsutil()

print("S1 chips:", len(glob.glob(f"{DATA}/S1Hand/*.tif")),
      "| labels:", len(glob.glob(f"{DATA}/LabelHand/*.tif")),
      "| split files:", sorted(os.listdir(f"{DATA}/flood_handlabeled")))
assert len(glob.glob(f"{DATA}/S1Hand/*.tif")) >= 446, "Download incomplete: rerun this cell"

## 2. Load all chips into memory (preprocessed once)

In [ ]:
import numpy as np
import pandas as pd
import rasterio

def chip_ids(word):
    files = [f for f in glob.glob(f"{DATA}/flood_handlabeled/*.csv") if word in os.path.basename(f).lower()]
    if word != "bolivia":
        files = [f for f in files if "bolivia" not in f.lower()]
    assert files, f"No split file containing '{word}'"
    names = pd.read_csv(files[0], header=None).iloc[:, 0].astype(str)
    return [re.sub(r"_(S1Hand|LabelHand|S2Hand|S1Weak).*$", "", os.path.basename(n)) for n in names]

def load_split(ids):
    X = np.empty((len(ids), 2, config.TILE_SIZE, config.TILE_SIZE), dtype=np.float16)
    Y = np.empty((len(ids), config.TILE_SIZE, config.TILE_SIZE), dtype=np.int8)
    for i, cid in enumerate(ids):
        with rasterio.open(f"{DATA}/S1Hand/{cid}_S1Hand.tif") as f:
            X[i] = preprocess_sar(f.read())
        with rasterio.open(f"{DATA}/LabelHand/{cid}_LabelHand.tif") as f:
            Y[i] = clean_label(f.read())
    return X, Y

ids = {s: chip_ids(s) for s in ["train", "valid", "test", "bolivia"]}
if QUICK_TEST:
    ids = {"train": ids["train"][:8], "valid": ids["valid"][:4], "test": ids["test"][:4], "bolivia": ids["bolivia"][:2]}
data = {s: load_split(v) for s, v in ids.items()}
for s, (X, Y) in data.items():
    print(f"{s:8s} {len(X):4d} chips | water share of labelled pixels: {(Y == 1).sum() / max((Y != -1).sum(), 1):.3f}")

## 3. Threshold baseline

One VV threshold chosen on the training data (Otsu's method): every pixel darker than it is called water.

In [ ]:
from skimage.filters import threshold_otsu

rng = np.random.default_rng(0)
Xtr, Ytr = data["train"]
vv = np.concatenate([rng.choice(x[0][y != -1].astype(np.float32), size=min(5000, int((y != -1).sum())), replace=False)
                     for x, y in zip(Xtr, Ytr) if (y != -1).sum() > 0])
THR_OTSU = float(threshold_otsu(vv))
print("Otsu threshold on scaled VV:", round(THR_OTSU, 3))

results = []   # every evaluation row ends up here
def baseline_eval(split):
    X, Y = data[split]
    c = Counts()
    for x, y in zip(X, Y):
        c.update(x[0].astype(np.float32) < THR_OTSU, y)
    return c.summary()

for split in ["valid", "test", "bolivia"]:
    results.append({"method": "threshold baseline", "split": split, **baseline_eval(split)})
pd.DataFrame(results)

## 4. Model, loss and training

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = device == "cuda"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

class Chips(Dataset):
    def __init__(self, X, Y, augment=False):
        self.X, self.Y, self.augment = X, Y, augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x, y = self.X[i].astype(np.float32), self.Y[i].astype(np.int64)
        if self.augment:   # random flips and 90-degree turns
            if np.random.rand() < 0.5: x, y = x[:, :, ::-1], y[:, ::-1]
            if np.random.rand() < 0.5: x, y = x[:, ::-1, :], y[::-1, :]
            k = np.random.randint(4); x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def worker_init(wid):
    np.random.seed((torch.initial_seed() + wid) % 2**32)

def bce_dice(logits, y):
    """BCE + Dice loss, both ignoring no-data pixels (label -1)."""
    logits = logits[:, 0].float()
    valid = (y != -1).float()
    t = y.clamp(min=0).float()
    bce = (F.binary_cross_entropy_with_logits(logits, t, reduction="none") * valid).sum() / valid.sum().clamp(min=1)
    p = torch.sigmoid(logits) * valid
    dice = 1 - (2 * (p * t).sum() + 1) / (p.sum() + (t * valid).sum() + 1)
    return bce + dice

def make_model():
    return smp.Unet(config.ENCODER, encoder_weights=PRETRAINED, in_channels=2, classes=1)

@torch.no_grad()
def predict_probs(model, X, batch=8):
    """Water probability for every chip, stored as float16 to save memory."""
    model.eval()
    out = np.empty((len(X),) + X.shape[2:], dtype=np.float16)
    for i in range(0, len(X), batch):
        xb = torch.from_numpy(X[i:i + batch].astype(np.float32)).to(device)
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            out[i:i + batch] = torch.sigmoid(model(xb)[:, 0].float()).cpu().numpy()
    return out

def score(probs, Y, thr=0.5):
    c, r = Counts(), Reliability()
    for p, y in zip(probs, Y):
        p = p.astype(np.float32)
        c.update(p > thr, y)
        r.update(p, y)
    return {**c.summary(), "ECE": round(r.ece(), 4)}

In [ ]:
def train_one(seed):
    """Train one model; keep the epoch with the best validation IoU. Resumes if already done."""
    path = f"{OUT}/models/unet_seed{seed}.pth"
    hist_path = f"{OUT}/history_seed{seed}.csv"
    if os.path.exists(path) and os.path.exists(hist_path):
        print(f"seed {seed}: already trained, loading")
        m = make_model().to(device); m.load_state_dict(torch.load(path, map_location=device)); return m

    seed_everything(seed)
    g = torch.Generator(); g.manual_seed(seed)
    tr = DataLoader(Chips(*data["train"], augment=True), batch_size=BATCH, shuffle=True,
                    num_workers=NUM_WORKERS, worker_init_fn=worker_init, generator=g, drop_last=True)
    model = make_model().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best, history, t0 = -1, [], time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train(); total = 0
        for x, y in tr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(x)
            loss = bce_dice(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            total += loss.item() * len(x)
        sched.step()
        val = score(predict_probs(model, data["valid"][0]), data["valid"][1])
        history.append({"epoch": epoch, "train_loss": total / (len(tr) * BATCH), "lr": opt.param_groups[0]["lr"], **val})
        if val["IoU"] > best:
            best = val["IoU"]; torch.save(model.state_dict(), path)
        mins = (time.time() - t0) / 60
        print(f"seed {seed} | epoch {epoch:2d}/{EPOCHS} | loss {history[-1]['train_loss']:.4f} | "
              f"val IoU {val['IoU']:.3f} | best {best:.3f} | {mins:.1f} min")

    pd.DataFrame(history).to_csv(hist_path, index=False)
    model.load_state_dict(torch.load(path, map_location=device))
    del opt, sched, scaler, tr
    return model

import gc
models = {}
for s in SEEDS:
    models[s] = train_one(s)
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
print("All models trained.")

## 5. Ensemble, best threshold and calibration (validation)

The ensemble probability is the average of the five models. The spread between models (standard deviation) is the uncertainty.

In [ ]:
val_probs = {s: predict_probs(m, data["valid"][0]) for s, m in models.items()}
Yv = data["valid"][1]
ens_val = np.mean([p.astype(np.float32) for p in val_probs.values()], axis=0)

# Best decision threshold for the ensemble, chosen on validation only
thr_scores = {round(float(t), 2): score(ens_val, Yv, t)["IoU"] for t in np.arange(0.20, 0.71, 0.05)}
BEST_THR = max(thr_scores, key=thr_scores.get)
print("Validation IoU by threshold:", thr_scores)
print("Best threshold:", BEST_THR)

for s, p in val_probs.items():
    results.append({"method": f"U-Net seed {s}", "split": "valid", **score(p, Yv)})
results.append({"method": "ensemble (0.5)", "split": "valid", **score(ens_val, Yv)})
results.append({"method": f"ensemble ({BEST_THR})", "split": "valid", **score(ens_val, Yv, BEST_THR)})
pd.DataFrame([r for r in results if r["split"] == "valid"])

## 6. Final test (run once)

The test set and the Bolivia set (a country never seen in training) are used only here, with the threshold chosen on validation.

In [ ]:
probs_store = {"valid": (val_probs, ens_val)}
for split in ["test", "bolivia"]:
    X, Y = data[split]
    member = {s: predict_probs(m, X) for s, m in models.items()}
    ens = np.mean([p.astype(np.float32) for p in member.values()], axis=0)
    probs_store[split] = (member, ens)
    for s, p in member.items():
        results.append({"method": f"U-Net seed {s}", "split": split, **score(p, Y)})
    results.append({"method": "ensemble (0.5)", "split": split, **score(ens, Y)})
    results.append({"method": f"ensemble ({BEST_THR})", "split": split, **score(ens, Y, BEST_THR)})

res = pd.DataFrame(results)
res.to_csv(f"{OUT}/results_all.csv", index=False)

# Summary: single models as mean +/- std across seeds, against the ensemble and the baseline
rows = []
for split in ["valid", "test", "bolivia"]:
    r = res[res.split == split]
    single = r[r.method.str.startswith("U-Net seed")]
    rows.append({"split": split,
                 "baseline IoU": r[r.method == "threshold baseline"].IoU.iloc[0],
                 "single U-Net IoU": f"{single.IoU.mean():.3f} +/- {single.IoU.std():.3f}",
                 "ensemble IoU": r[r.method == f"ensemble ({BEST_THR})"].IoU.iloc[0],
                 "single U-Net ECE": f"{single.ECE.mean():.4f}",
                 "ensemble ECE": r[r.method == "ensemble (0.5)"].ECE.iloc[0]})
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/results_summary.csv", index=False)
json.dump({"best_threshold": BEST_THR, "epochs": EPOCHS, "seeds": SEEDS, "otsu_threshold": THR_OTSU,
           "quick_test": QUICK_TEST}, open(f"{OUT}/run_info.json", "w"), indent=1)
summary

## 7. Figures

In [ ]:
import matplotlib.pyplot as plt

# Training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s in SEEDS:
    h = pd.read_csv(f"{OUT}/history_seed{s}.csv")
    ax[0].plot(h.epoch, h.train_loss, label=f"seed {s}")
    ax[1].plot(h.epoch, h.IoU, label=f"seed {s}")
ax[0].set_title("Training loss"); ax[1].set_title("Validation IoU")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/figures/training_curves.png", dpi=150); plt.show()

# Reliability diagram on the test set: single model (seed 0) vs ensemble
member, ens = probs_store["test"]
Yt = data["test"][1]
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for name, p in [("single U-Net (seed 0)", member[SEEDS[0]]), ("ensemble of 5", ens)]:
    r = Reliability()
    for pi, yi in zip(p, Yt): r.update(pi.astype(np.float32), yi)
    conf, acc, _ = r.curve()
    ax.plot(conf, acc, "o-", label=f"{name}, ECE {r.ece():.3f}")
ax.plot([0, 1], [0, 1], "--", c="gray", label="perfectly calibrated")
ax.set_xlabel("predicted probability of water"); ax.set_ylabel("observed share of water")
ax.set_title("Calibration on the test set"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/figures/reliability_test.png", dpi=150); plt.show()

# Example test chips: radar, label, ensemble probability, uncertainty
Xt = data["test"][0]
std = np.std([p.astype(np.float32) for p in member.values()], axis=0)
idx = np.argsort([-(y == 1).sum() for y in Yt])[:4]   # chips with the most water
fig, axes = plt.subplots(len(idx), 4, figsize=(13, 3.2 * len(idx)))
axes = np.atleast_2d(axes)
for row, i in zip(axes, idx):
    row[0].imshow(Xt[i][0].astype(np.float32), cmap="gray"); row[0].set_title(f"{ids['test'][i]} VV")
    row[1].imshow(np.where(Yt[i] == -1, 0.5, Yt[i]), cmap="gray", vmin=0, vmax=1); row[1].set_title("Label")
    row[2].imshow(ens[i], cmap="Blues", vmin=0, vmax=1); row[2].set_title("Ensemble water probability")
    row[3].imshow(std[i], cmap="magma", vmin=0, vmax=0.5); row[3].set_title("Uncertainty (spread)")
    for a in row: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/figures/test_examples.png", dpi=120); plt.show()
print("Saved figures to", f"{OUT}/figures")

## Done: what to send back

From the **Output** tab of this version, the key files are:
- `outputs/results_summary.csv`: the main comparison table
- `outputs/results_all.csv`: every model on every split
- `outputs/figures/`: training curves, reliability diagram, example predictions
- `outputs/models/`: the five trained models (needed for the Assam flood map)

Copy the summary table into `NOTES.md`, or share the Kaggle notebook link.